# Job Recommender — CV Skills → Ranked Jobs

**Pipeline**

1. Scraper produces `job_scrapper/jobs.json` (title, company, location, link, skills, ...).
2. User uploads a CV → an **N8N workflow** parses it and POSTs the extracted skills to FastAPI (`controller.py`).
3. This notebook loads the same jobs file, simulates the N8N payload, and uses the
   scoring helpers from `controller.py` (`normalize_skills`, `recommend_jobs`) to rank jobs.
4. The top recommendations are pushed back into the controller's in-memory cache so the
   `/webhook/get-recommendations` endpoint can serve them to the frontend.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Make the controller importable regardless of the working directory Jupyter picked
sys.path.append(str(Path.cwd()))

from controller import (
    CVSkillsPayload,
    extract_skills_from_profile,
    load_jobs,
    normalize_skills,
    recommend_jobs,
    score_job,
    send_recommendations_to_controller,
)

pd.set_option("display.max_colwidth", 80)

## 1. Load & inspect the scraped jobs

In [ ]:
jobs_df = load_jobs()
print(f"Loaded {len(jobs_df)} jobs")
print("Columns:", jobs_df.columns.tolist())
jobs_df[["title", "company", "location", "skills"]].head()

In [ ]:
# Quick sanity check: most common required skills across the dataset
from collections import Counter

skill_counts = Counter()
for skills in jobs_df["skills"].dropna():
    skill_counts.update(normalize_skills(skills))

pd.DataFrame(skill_counts.most_common(15), columns=["skill", "jobs_requiring_it"])

## 2. Simulate the N8N payload

The real N8N workflow posts JSON like:

```json
{ "skills": ["Python", "SQL", "Docker"], "preferred_locations": ["Tunis"], "top_n": 5 }
```

We validate the same shape here using the `CVSkillsPayload` pydantic model.

In [ ]:
sample_payload = CVSkillsPayload(
    skills=[
        "Python",
        "SQL",
        "Docker",
        "Machine Learning",
        "FastAPI",
        "Pandas",
        "NumPy",
        "Scikit-learn",
        "TensorFlow",
        "PyTorch",
        "Deep Learning",
        "NLP",
        "Computer Vision",
        "Data Analysis",
        "Data Visualization",
        "Matplotlib",
        "Seaborn",
        "Power BI",
        "Tableau",
        "Excel",
        "Git",
        "GitHub",
        "Linux",
        "Bash",
        "REST API",
        "GraphQL",
        "Flask",
        "Django",
        "Node.js",
        "JavaScript",
        "TypeScript",
        "React",
        "HTML",
        "CSS",
        "MongoDB",
        "PostgreSQL",
        "MySQL",
        "Redis",
        "Kafka",
        "Spark",
        "Hadoop",
        "Airflow",
        "ETL",
        "Data Engineering",
        "AWS",
        "Azure",
        "GCP",
        "Kubernetes",
        "CI/CD",
        "MLflow",
        "Agile",
        "Scrum",
    ],
    preferred_locations=["Tunis", "Sousse"],
    top_n=5,
)

user_skills = normalize_skills(sample_payload.skills)
print("Normalized user skills:", user_skills)

## 3. Score a single job (debugging helper)

In [ ]:
example_job = jobs_df.iloc[0]
print(f"{example_job['title']} @ {example_job['company']}")
print("Required:", normalize_skills(example_job["skills"]))
score_job(user_skills, normalize_skills(example_job["skills"]))

## 4. Rank all jobs for the user

In [ ]:
recommendations = recommend_jobs(
    user_skills=sample_payload.skills,
    jobs_df=jobs_df,
    top_n=sample_payload.top_n or 5,
    preferred_locations=sample_payload.preferred_locations,
)

recommendations[
    ["job_title", "company", "location", "match_score", "coverage", "skills_matched", "skills_missing"]
]

## 5. Human-readable summary of the top matches

In [ ]:
if recommendations.empty:
    print("No jobs matched the provided skills.")
else:
    for rank, row in enumerate(recommendations.itertuples(index=False), start=1):
        print("-" * 80)
        print(f"#{rank}  {row.job_title}  —  {row.company}")
        print(f"   Location : {row.location}")
        print(f"   Score    : {row.match_score:.1%}  (coverage {row.coverage:.0%}, jaccard {row.jaccard:.0%})")
        print(f"   Matched  : {', '.join(row.skills_matched) if row.skills_matched else '—'}")
        missing = row.skills_missing[:5]
        if missing:
            more = '' if len(row.skills_missing) <= 5 else f" (+{len(row.skills_missing)-5} more)"
            print(f"   Missing  : {', '.join(missing)}{more}")
        print(f"   Link     : {row.link}")

## 6. Push the recommendations back to the controller

The FastAPI app exposes `GET /webhook/get-recommendations`, which reads from the same
in-memory store we populate here. The N8N workflow (or the React frontend) can poll that
endpoint to fetch the latest ranked jobs.

In [ ]:
def recommendations_to_records(df: pd.DataFrame) -> list[dict]:
    """Make the DataFrame JSON-serializable (lists stay as lists)."""
    records = df.to_dict(orient="records")
    for rec in records:
        for key in ("skills_required", "skills_matched", "skills_missing"):
            if key in rec and rec[key] is None:
                rec[key] = []
    return records


records = recommendations_to_records(recommendations)
result = send_recommendations_to_controller(records)
result

## 7. End-to-end smoke test against the live API (optional)

Run `uvicorn controller:app --reload` from `job_recommendation/recommender`, then execute
the cell below to hit the endpoint the N8N workflow will call.

In [ ]:
# Uncomment to exercise the running FastAPI service:
#
# import requests
# response = requests.post(
#     "http://localhost:8000/webhook/cv-recommendations",
#     json=sample_payload.model_dump(),
#     timeout=10,
# )
# response.raise_for_status()
# response.json()